# Chapter 11 — Exercise Solutions## Training Agents with RL[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

### Solution 11.1: GAE Variance vs Group Size

In [ ]:
import numpy as np, matplotlib.pyplot as pltdef grpo_advantages(rewards):    r = np.array(rewards, dtype=float)    if r.std() < 1e-8: return np.zeros_like(r)    return (r - r.mean()) / (r.std() + 1e-8)def simulate_advantage_variance(p_correct, G_values, n_trials=2000):    variances = []    for G in G_values:        trial_vars = []        for _ in range(n_trials):            rewards = np.random.binomial(1, p_correct, G).astype(float)            adv = grpo_advantages(rewards)            trial_vars.append(adv.var())        variances.append(np.mean(trial_vars))    return variancesG_values = [2, 4, 8, 16, 32, 64]p_values = [0.1, 0.3, 0.5, 0.7, 0.9]fig, ax = plt.subplots(figsize=(10, 5))colors = plt.cm.viridis(np.linspace(0, 1, len(p_values)))for p, color in zip(p_values, colors):    var = simulate_advantage_variance(p, G_values)    ax.plot(G_values, var, 'o-', label=f'p_correct={p}', color=color, lw=2)ax.set_xlabel('Group Size G'); ax.set_ylabel('Mean Advantage Variance')ax.set_title('GRPO Advantage Variance vs Group Size and Task Difficulty')ax.legend(loc='upper right'); ax.grid(True, alpha=0.3)plt.tight_layout(); plt.show()print("Key findings:")print("1. Variance stabilises around G=8-16 for most task difficulties")print("2. Hardest tasks (p=0.1) need larger G to estimate advantages reliably")print("3. Easiest tasks (p=0.9) show low variance even at G=2 (rare failures)")print("4. Standard in practice: G=8 for most tasks, G=16 for hard tasks")print()print("G_values, p=0.5 variances:", dict(zip(G_values, [round(v,4) for v in simulate_advantage_variance(0.5, G_values)])))# YOUR TURN: Plot variance vs G for p=0.05 (very hard task — cold start problem)# At what G does variance stabilise? Does this motivate SFT warm-up?

### Solution 11.2: Format + Correctness Combined Reward

In [ ]:
import redef verify_math_answer(model_answer: str, correct_answer: str) -> float:    """Check if model answer is mathematically correct."""    try:        from sympy import sympify, simplify        model_expr   = sympify(model_answer.strip().strip('$'))        correct_expr = sympify(correct_answer)        return 1.0 if simplify(model_expr - correct_expr) == 0 else 0.0    except:        return 1.0 if model_answer.strip() == correct_answer.strip() else 0.0def format_reward(response: str) -> float:    """Reward presence of \\boxed{} format."""    pattern = r'\\boxed\{[^}]+\}'    return 1.0 if re.search(pattern, response) else 0.0def total_reward(response: str, correct_answer: str,                 w_correct: float = 0.8, w_format: float = 0.2) -> float:    """Combined reward: 80% correctness + 20% format compliance."""    # Extract answer from boxed if present    boxed_match = re.search(r'\\boxed\{([^}]+)\}', response)    answer_to_check = boxed_match.group(1) if boxed_match else response.strip()    c = verify_math_answer(answer_to_check, correct_answer)    f = format_reward(response)    return w_correct * c + w_format * f# Test casestest_cases = [    ("The answer is \\boxed{42}",   "42",  "Correct + formatted"),    ("The answer is 42",              "42",  "Correct, not formatted"),    ("The answer is \\boxed{43}",   "42",  "Wrong + formatted"),    ("Let me think... \\boxed{6*7}","42",  "Correct expression + formatted"),    ("I don't know",                  "42",  "Wrong, not formatted"),]print(f"{'Response':<40} {'Correct':>8} {'Format':>8} {'Total':>8}")print("-" * 68)for response, answer, label in test_cases:    c = verify_math_answer(re.search(r'\\boxed\{([^}]+)\}', response).group(1)                           if re.search(r'\\boxed\{([^}]+)\}', response) else response, answer)    f = format_reward(response)    t = 0.8*c + 0.2*f    print(f"{label:<40} {c:>8.1f} {f:>8.1f} {t:>8.2f}")# YOUR TURN: Add a third reward component for reasoning quality:# +0.1 if response contains "because" or "therefore" (reasoning words)# +0.1 if response has > 3 sentences (shows working)# Does this improve solution quality on hard problems?

### Solution 11.3: Reasoning Length vs Accuracy

In [ ]:
import numpy as np, matplotlib.pyplot as pltdef simulate_accuracy_vs_length(p_solve_per_token, lengths, n_problems=500):    """    Model: each token has probability p_solve_per_token of being the 'insight token'.    A problem is solved if at least one insight token appears in the available budget.    P(solve | L tokens) = 1 - (1-p)^L    """    return [1 - (1-p_solve_per_token)**L for L in lengths]lengths = [100, 200, 500, 1000, 2000, 4000, 8000]difficulties = {    'Easy (GSM8K)':    0.005,    # 5 per 1000 tokens    'Medium (MATH)':   0.001,    'Hard (AIME)':     0.0002,    'Expert':          0.00005,}fig, ax = plt.subplots(figsize=(10, 5))for label, p in difficulties.items():    accs = simulate_accuracy_vs_length(p, lengths)    ax.plot(lengths, accs, 'o-', label=label, lw=2)ax.set_xlabel('Reasoning Length (tokens)')ax.set_ylabel('Accuracy (P(solve))')ax.set_title('Accuracy vs Reasoning Length by Task Difficulty')ax.set_xscale('log')ax.legend(); ax.grid(True, alpha=0.3)plt.tight_layout(); plt.show()print("\nKey insights:")print("1. Easy tasks: even 200 tokens is sufficient (90%+ accuracy)")print("2. Hard tasks: need 2000-8000 tokens to achieve >50% accuracy")print("3. This is why o1 and R1 use adaptive compute: match budget to difficulty")print("4. RLVR training teaches the model WHEN to use long reasoning")print("   — short reasoning on easy problems, long on hard ones")for label, p in difficulties.items():    l_for_50 = int(np.log(0.5) / np.log(1-p))    print(f"{label:<20}: needs ~{l_for_50:5d} tokens for 50% accuracy")# YOUR TURN: Add noise to the model (token budget not perfectly used)# Simulate: each token is a reasoning token with prob 0.7, filler with prob 0.3# How does noise in reasoning quality affect the length-accuracy curve?